In [25]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import requests
from io import BytesIO
import time
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from bs4 import BeautifulSoup
from selenium.webdriver.common.action_chains import ActionChains
import pyautogui
import random
from tqdm import tqdm


In [26]:
url ="https://www.1001fonts.com/handwritten-fonts.html?page=1"
# Setup Chrome options
options = Options()

# options.binary_location = './chrome-mac-x64/Google Chrome for Testing.app/Contents/MacOS/Google Chrome for Testing'
options.binary_location ='/Applications/Google Chrome.app/Contents/MacOS/Google Chrome'

options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')


# https://www.educative.io/answers/how-to-use-a-specific-chrome-profile-in-python-selenium/
# options.add_argument(r"--user-data-dir=/Users/gimchangheon/Library/Application Support/Chromium")
# options.add_argument(r"--user-data-dir=/Users/gimchangheon/Library/Application Support/Google/Chrome")


# # #provide the profile name with which we want to open browser
# options.add_argument(r'--profile-directory=Profile 1')


webdriver_service = Service('./chromedriver-mac-x64/chromedriver')

driver = webdriver.Chrome(service=webdriver_service, options=options)

driver.set_window_position(0, 0)
# Open the webpage with Selenium

{'height': 1011, 'width': 1200, 'x': 0, 'y': 25}

In [30]:
base_url = "https://www.flaconi.de/locken-pflege/?offset="

base_url = "https://www.1001fonts.com/handwritten-fonts.html?page="
pages = range(1, 1218, 1)

page_links = [base_url + str(page) for page in pages]


In [31]:
from bs4 import BeautifulSoup
from tqdm import tqdm
import csv

data = []

# page_links=page_links[0:1]

# Iterate over each URL in the page_links list
for url in tqdm(page_links):
    # Open the webpage with Selenium
    driver.get(url)
    
    # Parse the page content using BeautifulSoup
    soup = BeautifulSoup(driver.page_source, 'html.parser')
    
    # Find the font list
    font_list = soup.find('ul', {'class': 'font-list', 'id': 'typeface-browsing-list'})
    if font_list:
        # Find all the font items
        font_items = font_list.find_all('li', {'class': 'font-list-item font-preview'})
        for item in font_items:
            # Extract the download link from the specific <a> tag
            download_link_tag = item.find('a', {'role': 'button', 'class': 'btn btn-success'})
            if download_link_tag and download_link_tag.has_attr('href'):
                link = download_link_tag.get('href')
                # Find the use type
                license_element = item.find('a', class_=lambda x: x and 'license-yes' in x)
                if license_element:
                    use_type = 'commercial use'
                else:
                    license_element = item.find('a', class_=lambda x: x and 'license-no' in x)
                    if license_element:
                        use_type = 'personal use'
                    else:
                        use_type = 'unknown'
                data.append({'link': link, 'use_type': use_type})
            else:
                # If the download link is not found, skip this item
                continue

# Save the data to a CSV file
with open('font_links_license.csv', 'w', newline='', encoding='utf-8') as csvfile:
    fieldnames = ['link', 'use_type']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    for entry in data:
        writer.writerow(entry)

# Optionally, print the data to verify
print(data)

100%|███████████████████████████████████████| 1217/1217 [39:31<00:00,  1.95s/it]

[{'link': '/download/scriptina.zip', 'use_type': 'commercial use'}, {'link': '/download/creattion-demo.zip', 'use_type': 'personal use'}, {'link': '/download/great-vibes.zip', 'use_type': 'commercial use'}, {'link': '/download/alex-brush.zip', 'use_type': 'commercial use'}, {'link': '/download/chopin-script.zip', 'use_type': 'commercial use'}, {'link': '/download/autumn-in-november.zip', 'use_type': 'personal use'}, {'link': '/download/southam-demo.zip', 'use_type': 'personal use'}, {'link': '/download/mf-i-love-glitter.zip', 'use_type': 'personal use'}, {'link': '/download/youmurderer-bb.zip', 'use_type': 'personal use'}, {'link': '/download/nexa-rust.zip', 'use_type': 'commercial use'}, {'link': '/download/quigleywiggly.zip', 'use_type': 'commercial use'}, {'link': '/download/billy-argel-font.zip', 'use_type': 'personal use'}, {'link': '/download/candy-inc.zip', 'use_type': 'personal use'}, {'link': '/download/sacramento.zip', 'use_type': 'commercial use'}, {'link': '/download/monsie

In [ ]:
now I would like to find this one.
There are two types  "commercial use" and "personal use"

<a href="/scriptina-font.html#license" class="btn btn-link license-yes me-2" data-bs-toggle="tooltip" data-bs-html="true" aria-disabled="true" aria-label="This font is free for<br>commercial use" data-bs-original-title="This font is free for<br>commercial use"><i class="fa fo-money"></i> </a>

<a href="/billy-argel-font-font.html#license" class="btn btn-link license-no me-2" data-bs-toggle="tooltip" data-bs-html="true" aria-disabled="true" aria-label="This font is free for<br>personal use" data-bs-original-title="This font is free for<br>personal use"><i class="fa fo-money"></i> </a>

So I'd like to map each link to this type "commercial use" or "personal use"

All fonts are under here : 
<ul class="font-list" id="typeface-browsing-list">

and all the font items are under here : 
<li class="font-list-item font-preview">